<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/02_llms_em_softwares/hands_on_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho Final — LLMs em Sistemas de Software

> **Instruções Gerais**
>
> Este trabalho deve ser entregue como um **Jupyter Notebook (.ipynb)** executado no **Google Colab**.
> Todas as células de código devem estar executadas e com os outputs visíveis no momento da entrega.
>
> - Documente seu raciocínio em células Markdown ao longo do notebook.
> - Use comentários no código para explicar decisões de implementação relevantes.
> - Certifique-se de que o notebook pode ser executado de ponta a ponta sem erros.
> - Variáveis sensíveis (API keys) devem ser carregadas via `google.colab.userdata` ou variáveis de ambiente — **nunca** expostas diretamente no código.

---
---
---

## Tema do Trabalho

Você irá construir um **sistema de perguntas e respostas baseado em RAG (Retrieval-Augmented Generation)** sobre uma base de documentos à sua escolha (exemplos: artigos técnicos, legislação, documentação de software, relatórios, etc.). O sistema deve ser construído com **LangChain**, explorar diferentes estratégias de **prompt engineering** e ser avaliado com métricas objetivas.


---
---
---


## Parte 1 — Prompt Engineering (3 pontos)

### Objetivo
Demonstrar domínio prático de diferentes técnicas de prompting e compreender como os hiperparâmetros do modelo afetam as respostas geradas.

### Tarefas

**1.1 — Comparação de estratégias de prompting**

Escolha uma tarefa de sua preferência (ex: classificação de texto, extração de informação, sumarização, geração de código) e implemente as três estratégias abaixo para a **mesma tarefa e o mesmo modelo**:

- **Zero-shot**: apenas a instrução, sem exemplos.
- **Few-shot**: instrução com 3 exemplos no prompt.
- **Chain-of-Thought (CoT)**: instrução que induz o modelo a raciocinar passo a passo antes de responder.

Para cada estratégia, execute ao menos **3 inputs diferentes** e registre as respostas.

**1.2 — Análise dos hiperparâmetros**

Usando a mesma tarefa do item 1.1, varie os seguintes parâmetros e registre o impacto observado nas respostas:

| Parâmetro     | Valores a testar         |
|---------------|--------------------------|
| `temperature` | 0.0 / 0.7 / 1.4          |
| `top_p`       | 0.5 / 0.9 / 1.0          |
| `max_tokens`  | Restritivo / Adequado / Amplo |

**1.3 — Discussão**

Em uma célula Markdown, responda:
- Qual estratégia apresentou os melhores resultados para a sua tarefa? Por quê?
- Como a variação de `temperature` impactou a consistência e criatividade das respostas?
- Quais limitações você identificou em cada abordagem?

### Critérios de Avaliação
- Implementação correta das três estratégias (1 pt)
- Experimento com hiperparâmetros documentado e com outputs visíveis (1 pt)
- Qualidade e profundidade da análise crítica (1 pt)


In [ ]:
# Parte 1 - Setup geral (reutilizado nas Partes 2 e 3)
!pip -q install groq langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface faiss-cpu sentence-transformers rouge-score pypdf pandas matplotlib

import os
import json
from textwrap import dedent
import pandas as pd
from groq import Groq
from IPython.display import display, Markdown

# Carrega chave de forma segura (Colab Secrets) com fallback para variavel de ambiente
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Defina GROQ_API_KEY no Colab Secrets ou nas variaveis de ambiente.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.1-8b-instant"

def chat_groq(user_prompt, system_prompt="Voce e um assistente tecnico e objetivo.", temperature=0.2, top_p=1.0, max_tokens=300):
    """Chamada padrao ao modelo para manter reproducibilidade dos experimentos."""
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return completion.choices[0].message.content.strip()

# Tarefa escolhida: classificacao de intencao em atendimento de e-commerce
labels = [
    "pagamento",
    "entrega",
    "troca_devolucao",
    "fidelidade_promocoes",
]

inputs_teste = [
    "Posso pagar por boleto no site oficial?",
    "Fiz compra com entrega expressa. Consigo aplicar cupom de frete gratis?",
    "Troquei um item no e-commerce e quero trocar de novo em loja fisica. Pode?",
]

few_shot_exemplos = dedent("""
Exemplo 1
Pergunta: Quais cartoes aceitos no site?
Rotulo: pagamento
Justificativa curta: Pergunta sobre forma de pagamento no e-commerce.

Exemplo 2
Pergunta: Qual prazo da entrega super expressa?
Rotulo: entrega
Justificativa curta: Pergunta sobre modalidade e prazo de entrega.

Exemplo 3
Pergunta: Em quanto tempo posso devolver um produto online?
Rotulo: troca_devolucao
Justificativa curta: Pergunta sobre politica de devolucao.
""")

def prompt_zero_shot(pergunta):
    return dedent(f"""
    Classifique a pergunta em exatamente um rotulo: {labels}.
    Responda no formato JSON com as chaves: rotulo, justificativa.

    Pergunta: {pergunta}
    """)

def prompt_few_shot(pergunta):
    return dedent(f"""
    Classifique a pergunta em exatamente um rotulo: {labels}.
    Siga o padrao dos exemplos.
    Responda em JSON com: rotulo, justificativa.

    {few_shot_exemplos}
   
    Agora classifique:
    Pergunta: {pergunta}
    """)

def prompt_cot(pergunta):
    return dedent(f"""
    Classifique a pergunta em exatamente um rotulo: {labels}.
    Primeiro descreva seu raciocinio passo a passo em 3 passos curtos.
    Depois entregue um JSON final com: rotulo, justificativa.

    Pergunta: {pergunta}
    """)

print("Setup concluido.")
print(f"Modelo em uso: {MODEL_NAME}")
print(f"Total de entradas de teste: {len(inputs_teste)}")

In [ ]:
# 1.1 - Comparacao entre Zero-shot, Few-shot e Chain-of-Thought (CoT)
estrategias = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "chain_of_thought": prompt_cot,
}

registros = []
for pergunta in inputs_teste:
    for nome_estrategia, fn_prompt in estrategias.items():
        resposta = chat_groq(
            fn_prompt(pergunta),
            system_prompt="Voce e um classificador de intencao para e-commerce.",
            temperature=0.2,
            top_p=1.0,
            max_tokens=350,
        )
        registros.append({
            "pergunta": pergunta,
            "estrategia": nome_estrategia,
            "resposta_modelo": resposta,
        })

df_prompting = pd.DataFrame(registros)
display(Markdown("### Resultados - Estrategias de Prompting"))
display(df_prompting)

for _, row in df_prompting.iterrows():
    print("=" * 100)
    print(f"Pergunta: {row['pergunta']}")
    print(f"Estrategia: {row['estrategia']}")
    print("Resposta:")
    print(row["resposta_modelo"])

In [ ]:
# 1.2 - Analise de hiperparametros na mesma tarefa
pergunta_base = "Tenho cashback. Posso usar todo o saldo em qualquer compra?"

cenarios = [
    {"parametro": "temperature", "valor": "0.0", "temperature": 0.0, "top_p": 1.0, "max_tokens": 220},
    {"parametro": "temperature", "valor": "0.7", "temperature": 0.7, "top_p": 1.0, "max_tokens": 220},
    {"parametro": "temperature", "valor": "1.4", "temperature": 1.4, "top_p": 1.0, "max_tokens": 220},
    {"parametro": "top_p", "valor": "0.5", "temperature": 0.7, "top_p": 0.5, "max_tokens": 220},
    {"parametro": "top_p", "valor": "0.9", "temperature": 0.7, "top_p": 0.9, "max_tokens": 220},
    {"parametro": "top_p", "valor": "1.0", "temperature": 0.7, "top_p": 1.0, "max_tokens": 220},
    {"parametro": "max_tokens", "valor": "restritivo(60)", "temperature": 0.7, "top_p": 1.0, "max_tokens": 60},
    {"parametro": "max_tokens", "valor": "adequado(220)", "temperature": 0.7, "top_p": 1.0, "max_tokens": 220},
    {"parametro": "max_tokens", "valor": "amplo(500)", "temperature": 0.7, "top_p": 1.0, "max_tokens": 500},
]

registros_hp = []
for c in cenarios:
    resposta = chat_groq(
        prompt_zero_shot(pergunta_base),
        system_prompt="Voce e um classificador de intencao para e-commerce.",
        temperature=c["temperature"],
        top_p=c["top_p"],
        max_tokens=c["max_tokens"],
    )
    registros_hp.append({
        "parametro": c["parametro"],
        "valor_testado": c["valor"],
        "temperature": c["temperature"],
        "top_p": c["top_p"],
        "max_tokens": c["max_tokens"],
        "tamanho_resposta_chars": len(resposta),
        "resposta_modelo": resposta,
    })

df_hiperparametros = pd.DataFrame(registros_hp)
display(Markdown("### Resultados - Analise de Hiperparametros"))
display(df_hiperparametros[[
    "parametro", "valor_testado", "temperature", "top_p", "max_tokens", "tamanho_resposta_chars"
]])

display(Markdown("### Amostra de Respostas"))
for _, row in df_hiperparametros.iterrows():
    print("-" * 100)
    print(f"{row['parametro']} = {row['valor_testado']}")
    print(row["resposta_modelo"])

### 1.3 Discussao Critica [PREENCHER MANUALMENTE]

Preencha esta secao apos analisar os outputs gerados acima:

- Melhor estrategia para a tarefa (zero-shot, few-shot ou CoT) e justificativa tecnica.
- Impacto de temperature na consistencia versus criatividade.
- Limites observados em cada abordagem.
- Evidencias concretas citando exemplos das respostas obtidas.

## Parte 2 — Construção do Pipeline RAG com LangChain (3,5 pontos)

### Objetivo
Construir um pipeline RAG completo e funcional utilizando LangChain, integrando carregamento de documentos, chunking, embeddings, banco vetorial e recuperação.

### Tarefas

**2.1 — Preparação da base de dados**

- Escolha uma coleção de documentos (mínimo de **3 arquivos** em PDF, TXT ou CSV).
- Carregue os documentos utilizando os **Document Loaders** do LangChain.
- Aplique **uma estratégia de chunking** (ex: `RecursiveCharacterTextSplitter` com tamanhos distintos, ou `CharacterTextSplitter` ou chunking por parágrafo) e justifique a escolha final.

**2.2 — Indexação vetorial**

- Gere embeddings com um modelo à sua escolha (ex: OpenAI `text-embedding-ada-002`, HuggingFace `sentence-transformers`, etc.).
- Indexe os chunks em um banco vetorial (FAISS, Chroma ou Pinecone).
- Demonstre uma busca de similaridade direta no banco vetorial com ao menos **2 queries de teste**, exibindo os chunks recuperados.

**2.3 — Pipeline de geração**

- Monte uma chain de RAG com LangChain (`RetrievalQA` ou `ConversationalRetrievalChain`) que:
  - Recupere os `top-k` chunks mais relevantes (experimente ao menos dois valores de `k`).
  - Injete o contexto recuperado em um prompt estruturado.
  - Gere a resposta final com o LLM de sua escolha.
- Execute ao menos **3 perguntas** sobre a sua base de documentos e exiba as respostas com os respectivos trechos de contexto utilizados.

### Critérios de Avaliação
- Carregamento, chunking e indexação corretos e justificados (1,5 pt)
- Pipeline RAG funcional com contexto injetado corretamente (1,5 pt)
- Diversidade de queries testadas e clareza nos outputs (0,5 pt)


## Parte 2 - Construcao do Pipeline RAG com LangChain

Arquitetura adotada (alinhada ao conteudo da trilha 02_llms_em_softwares):

- Framework: LangChain para orquestracao modular de loaders, splitter, retriever e chain de QA.
- Banco vetorial: FAISS pela simplicidade e performance local para prototipagem/reproducao.
- Embeddings: modelo multilingual Sentence-Transformers para lidar com conteudo em portugues.
- Geracao: modelo Groq com baixa latencia para iteracao rapida e custo controlado.

Justificativa tecnica: essa combinacao segue o padrao Retrieval-Augmented Generation visto em C_arquitetura_rag e a filosofia de pipelines modulares apresentada em B_framework_langchain.

## Tarefa 2.1 - Preparacao da base de dados

Nesta etapa utilizamos apenas a base de conhecimento do projeto:

- knowlegde_base/revenda.txt
- knowlegde_base/preco_estoque.csv
- knowlegde_base/faq_ecommerce.pdf

A decisao de usar multiplos formatos (TXT, CSV, PDF) permite validar o pipeline com dados heterogeneos, como discutido em C_arquitetura_rag (ingestao e retrieval).

In [ ]:
from pathlib import Path

base_kb = Path("knowlegde_base")
arquivos_kb = sorted([p.name for p in base_kb.glob("*") if p.is_file()])

print("Arquivos encontrados na base de conhecimento:")
for a in arquivos_kb:
    print(f"- {a}")

tipos = sorted({p.suffix.lower() for p in base_kb.glob('*') if p.is_file()})
print(f"\nTipos detectados: {tipos}")
assert ".txt" in tipos and ".csv" in tipos and ".pdf" in tipos, "A base deve conter .txt, .csv e .pdf"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### 2.1.1 Reuso de configuracao do modelo

Reaproveitamos o mesmo cliente/modelo configurado na Parte 1 para manter consistencia experimental e evitar variaveis de confusao na avaliacao final.

In [ ]:
# Configuracao defensiva: recria cliente apenas se necessario
if "GROQ_API_KEY" not in globals() or not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    except Exception:
        GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY nao encontrada. Configure antes de continuar.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

if "client" not in globals():
    client = Groq(api_key=GROQ_API_KEY)

if "MODEL_NAME" not in globals():
    MODEL_NAME = "llama-3.1-8b-instant"

print("Configuracao pronta para o pipeline RAG.")
print(f"Modelo de geracao: {MODEL_NAME}")

✅ Configuração Groq OK!
   Model : llama-3.1-8b-instant


### 2.1.2 Preparacao e ingestao dos documentos

Descricao da estrategia de ingestao:

- TXT com TextLoader (conteudo descritivo de revenda).
- CSV com CSVLoader (dados estruturados de preco/estoque).
- PDF com PyPDFLoader (FAQ de e-commerce).

Essa composicao cobre dados narrativos e tabulares, melhorando a capacidade de resposta do RAG para perguntas comerciais e operacionais.

### 2.1.3 Carregamento dos arquivos com Document Loaders

In [ ]:
from langchain_community.document_loaders import TextLoader, CSVLoader, PyPDFLoader

txt_docs, csv_docs, pdf_docs = [], [], []

for fp in sorted(base_kb.glob("*.txt")):
    docs = TextLoader(str(fp), encoding="utf-8").load()
    for d in docs:
        d.metadata["tipo_fonte"] = "txt"
    txt_docs.extend(docs)

for fp in sorted(base_kb.glob("*.csv")):
    docs = CSVLoader(file_path=str(fp), encoding="utf-8").load()
    for d in docs:
        d.metadata["tipo_fonte"] = "csv"
    csv_docs.extend(docs)

for fp in sorted(base_kb.glob("*.pdf")):
    docs = PyPDFLoader(str(fp)).load()
    for d in docs:
        d.metadata["tipo_fonte"] = "pdf"
    pdf_docs.extend(docs)

documentos = txt_docs + csv_docs + pdf_docs

print(f"TXT carregados: {len(txt_docs)}")
print(f"CSV carregados: {len(csv_docs)}")
print(f"PDF (paginas) carregadas: {len(pdf_docs)}")
print(f"Total de documentos LangChain: {len(documentos)}")

for i, doc in enumerate(documentos[:5], start=1):
    source = doc.metadata.get("source", "sem_fonte")
    tipo = doc.metadata.get("tipo_fonte", "desconhecido")
    print("-" * 90)
    print(f"Doc {i} | tipo={tipo} | source={source}")
    print(doc.page_content[:220].replace("\n", " ") + "...")

3 documentos criados na pasta 'documentos/':
    historia_ia.txt
    tipos_ia.txt
    aplicacoes_etica_ia.txt


### 2.1.4 Justificativa e configuracao do chunking

Usamos RecursiveCharacterTextSplitter com overlap por ser uma estrategia robusta para dados heterogeneos (TXT/CSV/PDF), preservando mais contexto entre chunks adjacentes e reduzindo perda de informacao em limites de corte.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

configs_chunking = [
    {"nome": "A_curto", "chunk_size": 300, "chunk_overlap": 40},
    {"nome": "B_medio", "chunk_size": 600, "chunk_overlap": 80},
    {"nome": "C_longo", "chunk_size": 900, "chunk_overlap": 120},
]

def gerar_chunks(cfg, docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        separators=["\n\n", "\n", " ", ""],
        length_function=len,
    )
    return splitter.split_documents(docs)

  Total de documentos carregados: 3

   Documento 1:
   Arquivo: documentos/historia_ia.txt
   Tamanho: 1150 caracteres
   Prévia:  Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da...

   Documento 2:
   Arquivo: documentos/tipos_ia.txt
   Tamanho: 1389 caracteres
   Prévia:  Tipos de Inteligência Artificial

A IA pode ser classificada em diferentes categorias com base em su...

   Documento 3:
   Arquivo: documentos/aplicacoes_etica_ia.txt
   Tamanho: 1668 caracteres
   Prévia:  Aplicações e Ética da Inteligência Artificial

Aplicações Práticas da IA:

Saúde: A IA é utilizada p...



### 2.1.5 Teste de tamanhos de chunk e escolha final

In [ ]:
resumo_chunking = []
chunks_por_config = {}

for cfg in configs_chunking:
    ch = gerar_chunks(cfg, documentos)
    chunks_por_config[cfg["nome"]] = ch
    tamanhos = [len(x.page_content) for x in ch]
    resumo_chunking.append({
        "config": cfg["nome"],
        "chunk_size": cfg["chunk_size"],
        "overlap": cfg["chunk_overlap"],
        "qtd_chunks": len(ch),
        "tam_medio": round(sum(tamanhos) / len(tamanhos), 1),
        "tam_max": max(tamanhos),
    })

df_chunking = pd.DataFrame(resumo_chunking).sort_values("qtd_chunks")
display(df_chunking)

# Escolha final: configuracao media (equilibrio entre granularidade e contexto)
config_escolhida = next(c for c in configs_chunking if c["nome"] == "B_medio")
chunks = chunks_por_config[config_escolhida["nome"]]

print("\nConfiguracao final escolhida:")
print(config_escolhida)
print(f"Total de chunks finais: {len(chunks)}")

In [ ]:
print("Previa de chunks (amostra):")
for i, ch in enumerate(chunks[:6], start=1):
    print("=" * 90)
    print(f"Chunk {i}")
    print(f"Fonte: {ch.metadata.get('source', 'sem_fonte')}")
    print(f"Tipo: {ch.metadata.get('tipo_fonte', 'desconhecido')}")
    print(f"Tamanho: {len(ch.page_content)}")
    print(ch.page_content[:260].replace("\n", " ") + "...")

Chunking concluído!
   Documentos originais: 3
   Chunks gerados: 12
   Tamanho configurado: 500 caracteres (com 50 de overlap)

PRÉVIA DOS PRIMEIROS 5 CHUNKS:

Chunk 1 (fonte: documentos/historia_ia.txt):
   Tamanho: 331 caracteres
   Conteúdo:
   Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da computação que busca
criar sistemas capazes de realizar tarefas que normalmente requerem inteligênc...
----------------------------------------------------------------------

Chunk 2 (fonte: documentos/historia_ia.txt):
   Tamanho: 245 caracteres
   Conteúdo:
   Nas décadas de 1960 e 1970, os pesquisadores desenvolveram sistemas especialistas,
que eram programas baseados em regras lógicas para resolver problemas específicos.
No entanto, essas abordagens tinha...
----------------------------------------------------------------------

Chunk 3 (fonte: documentos/historia_ia.txt):
   Tamanho: 288 caracteres
   Conteúdo:
   O chamado "inverno da I

## Tarefa 2.2 - Indexacao Vetorial

Nesta etapa convertemos os chunks em embeddings e indexamos no FAISS para busca semantica de alta eficiencia. A escolha segue o material de arquitetura RAG (retriever + gerador) e pipelines modulares do modulo de LangChain.

### 2.2.1 Embeddings multilingual + banco vetorial FAISS

Embeddings escolhidos: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2. Motivo: modelo leve, boa cobertura semantica em portugues e custo computacional baixo para ambiente Colab/CPU.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
modelo_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

banco_vetorial = FAISS.from_documents(
    documents=chunks,
    embedding=modelo_embeddings,
    distance_strategy="COSINE",
)

print("Indexacao concluida.")
print(f"Total de chunks indexados: {len(chunks)}")
print("Modelo de embedding: paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Carregando modelo de embeddings multilíngue...
   Banco vetorial FAISS criado com sucesso!
   Total de vetores indexados: 12
   Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2


### 2.2.2 Demonstracao de busca por similaridade no FAISS

Executamos duas queries para validar retrieval semantico. Como usamos similaridade por cosseno, valores mais altos indicam maior relevancia.

In [ ]:
queries_teste = [
    "Quais formas de pagamento sao aceitas no e-commerce do Boticario?",
    "Como funciona troca em loja fisica para compra online?",
]

for q in queries_teste:
    print("=" * 100)
    print(f"QUERY: {q}")
    resultados = banco_vetorial.similarity_search_with_relevance_scores(q, k=3)
    for i, (doc, score) in enumerate(resultados, start=1):
        print("-" * 90)
        print(f"Top {i} | score={score:.4f}")
        print(f"Fonte: {doc.metadata.get('source', 'sem_fonte')}")
        print(doc.page_content[:260].replace("\n", " ") + "...")

🔍 QUERY 1: "Quando surgiu o termo Inteligência Artificial?"

📌 Resultado 1 (distância L2: 0.3410):
   Fonte: documentos/historia_ia.txt
   Conteúdo: Inteligência Artificial: Uma Breve História

A Inteligência Artificial (IA) é um campo da ciência da computação que busca
criar sistemas capazes de realizar tarefas que normalmente requerem inteligência
humana. O termo foi cunhado por John McCarthy e...
----------------------------------------------------------------------

📌 Resultado 2 (distância L2: 0.6569):
   Fonte: documentos/historia_ia.txt
   Conteúdo: O chamado "inverno da IA" ocorreu nos anos 1980 e início dos anos 1990, quando o
financiamento e o interesse na área diminuíram drasticamente devido às expectativas
não cumpridas. A recuperação veio com o avanço do poder computacional e o
surgimento ...
----------------------------------------------------------------------

📌 Resultado 3 (distância L2: 0.7205):
   Fonte: documentos/tipos_ia.txt
   Conteúdo: 2. IA Geral (AGI - Artific

## Tarefa 2.3 - Pipeline de geracao (RetrievalQA)

Implementacao com LangChain RetrievalQA (chain_type=stuff) para injetar contexto recuperado no prompt e gerar resposta final. Testamos k=3 e k=5 para comparar cobertura versus ruido de contexto.

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

prompt_rag = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Voce e um assistente de atendimento do Boticario. "
        "Responda apenas com base no contexto abaixo. "
        "Se a informacao nao estiver no contexto, diga explicitamente que nao encontrou.\n\n"
        "Contexto:\n{context}\n\n"
        "Pergunta: {question}\n\n"
        "Resposta objetiva:"
    ),
)

llm_rag = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    max_tokens=350,
)

def construir_rag_qa(k):
    retriever = banco_vetorial.as_retriever(search_kwargs={"k": k})
    return RetrievalQA.from_chain_type(
        llm=llm_rag,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt_rag},
    )

perguntas_rag = [
    "Quais bandeiras de cartao sao aceitas no e-commerce?",
    "O cliente consegue boleto para compra comum no varejo online?",
    "Como funciona a regra de troca quando o item novo tem valor menor?",
]

resultados_rag = []
for k in [3, 5]:
    qa_chain = construir_rag_qa(k)
    for pergunta in perguntas_rag:
        out = qa_chain.invoke({"query": pergunta})
        resposta = out["result"]
        fontes = [d.metadata.get("source", "sem_fonte") for d in out["source_documents"]]
        contexto_curto = [d.page_content[:180].replace("\n", " ") + "..." for d in out["source_documents"]]

        resultados_rag.append({
            "k": k,
            "pergunta": pergunta,
            "resposta": resposta,
            "fontes": " | ".join(sorted(set(fontes))),
            "contexto_top": contexto_curto,
        })

df_rag = pd.DataFrame(resultados_rag)
display(df_rag[["k", "pergunta", "resposta", "fontes"]])

for _, row in df_rag.iterrows():
    print("=" * 110)
    print(f"k={row['k']} | Pergunta: {row['pergunta']}")
    print(f"Resposta: {row['resposta']}")
    print("Contexto utilizado (amostra):")
    for c in row["contexto_top"]:
        print(f"- {c}")

### Analise Qualitativa do RAG [PREENCHER MANUALMENTE]

Preencha apos inspecionar as respostas e contextos recuperados:

- Para quais perguntas k=3 foi suficiente?
- Em quais casos k=5 melhorou completude ou aumentou ruido?
- Houve sinais de alucinacao? Cite exemplos.
- Julgamento geral da qualidade das respostas (clareza, precisao, fidelidade ao contexto).

## Parte 3 — Avaliação do Sistema (3,5 pontos)

### Objetivo
Avaliar a qualidade do sistema RAG construído utilizando métricas quantitativas e qualitativas.

### Tarefas

**3.1 — Conjunto de avaliação**

- Crie um conjunto de **10 pares (pergunta, resposta esperada)** manualmente anotados, com base nos documentos da sua base. Organize-os em um dicionário ou DataFrame.

**3.2 — Métricas automáticas**

Para cada par do conjunto de avaliação, calcule:

- **ROUGE-L** entre a resposta gerada e a resposta esperada.
- **Similaridade semântica** (cosseno) entre os embeddings da resposta gerada e da resposta esperada.

Apresente os resultados em uma tabela e calcule a média geral.

**3.3 — LLM-as-a-judge**

- Implemente um avaliador baseado em LLM que, para cada resposta gerada, atribua uma nota de **1 a 5** (escala Likert) nos seguintes critérios:
  - **Fidelidade ao contexto** (*groundedness*): a resposta está ancorada nos documentos recuperados?
  - **Relevância**: a resposta responde de fato à pergunta?
  - **Completude**: a resposta abrange os pontos principais?
- Exiba a distribuição das notas em um gráfico.

### Critérios de Avaliação
- Conjunto de avaliação bem definido e métricas automáticas calculadas corretamente (1,75 pt)
- LLM-as-a-judge implementado com critérios claros e distribuição visualizada (1,75 pt)


In [ ]:
# 3.1 - Conjunto de avaliacao com 10 pares (pergunta, resposta esperada)
dataset_avaliacao = [
    {
        "pergunta": "Quais formas de pagamento o e-commerce aceita?",
        "resposta_esperada": "O e-commerce aceita PIX e cartao de credito (Visa, MasterCard, Amex, Diners, Elo e Hipercard).",
    },
    {
        "pergunta": "Em quantas vezes da para parcelar no cartao sem juros?",
        "resposta_esperada": "E possivel parcelar em ate 10x sem juros, com parcela minima de R$ 20,00.",
    },
    {
        "pergunta": "Qual o prazo para pagar um PIX no checkout?",
        "resposta_esperada": "O PIX vence em 30 minutos; apos isso o pedido e cancelado e deve ser feito novamente.",
    },
    {
        "pergunta": "Boleto e aceito no e-commerce de varejo?",
        "resposta_esperada": "Nao. Para varejo online, as formas sao PIX e cartao; boleto e restrito a canais especificos.",
    },
    {
        "pergunta": "Entrega Super Expressa aceita frete gratis ou cupom de isencao?",
        "resposta_esperada": "Nao. A Entrega Super Expressa nao acumula frete gratis nem cupons de isencao de frete.",
    },
    {
        "pergunta": "No Clique e Retire, o cliente paga frete?",
        "resposta_esperada": "Nao, o Clique e Retire nao cobra frete nem taxa de manuseio.",
    },
    {
        "pergunta": "Qual documento e exigido para retirar pedido em loja no Clique e Retire?",
        "resposta_esperada": "E obrigatorio apresentar documento oficial com foto; ha validacao digital com imagem e assinatura.",
    },
    {
        "pergunta": "Qual o prazo para arrependimento/devolucao de compra online?",
        "resposta_esperada": "O prazo informado e de ate 30 dias corridos apos o recebimento, com produto sem uso e com nota fiscal.",
    },
    {
        "pergunta": "Na troca em loja, se o novo item for mais barato, a loja devolve troco?",
        "resposta_esperada": "Nao. Nao ha troco em dinheiro; e preciso complementar com outros itens para igualar ou superar o valor.",
    },
    {
        "pergunta": "Como funciona o limite de resgate de cashback?",
        "resposta_esperada": "O resgate e limitado a ate 10% da nova compra e o uso e unico/parcial por CPF, consumindo o saldo residual.",
    },
]

df_dataset = pd.DataFrame(dataset_avaliacao)
display(Markdown("### Dataset de avaliacao (10 pares)"))
display(df_dataset)

qa_eval = construir_rag_qa(k=5)

predicoes = []
for item in dataset_avaliacao:
    out = qa_eval.invoke({"query": item["pergunta"]})
    resposta = out["result"]
    fontes = [d.metadata.get("source", "sem_fonte") for d in out["source_documents"]]
    contexto = "\n".join([d.page_content[:250] for d in out["source_documents"]])

    predicoes.append({
        "pergunta": item["pergunta"],
        "resposta_esperada": item["resposta_esperada"],
        "resposta_gerada": resposta,
        "fontes": " | ".join(sorted(set(fontes))),
        "contexto_recuperado": contexto,
    })

df_eval = pd.DataFrame(predicoes)
display(Markdown("### Respostas geradas pelo RAG (k=5)"))
display(df_eval[["pergunta", "resposta_gerada", "fontes"]])

In [ ]:
# 3.2 - Metricas automaticas: ROUGE-L e similaridade semantica (cosseno)
import numpy as np
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a, dtype=float)
    b = np.array(vec_b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

rouge_scores = []
semantic_scores = []

for _, row in df_eval.iterrows():
    ref = row["resposta_esperada"]
    gen = row["resposta_gerada"]

    rouge_l_f1 = scorer.score(ref, gen)["rougeL"].fmeasure
    rouge_scores.append(rouge_l_f1)

    emb_ref = modelo_embeddings.embed_query(ref)
    emb_gen = modelo_embeddings.embed_query(gen)
    sem = cosine_similarity(emb_ref, emb_gen)
    semantic_scores.append(sem)

df_eval["rougeL_f1"] = rouge_scores
df_eval["similaridade_semantica"] = semantic_scores

display(Markdown("### Tabela de metricas automaticas"))
display(df_eval[["pergunta", "rougeL_f1", "similaridade_semantica"]])

media_rouge = float(df_eval["rougeL_f1"].mean())
media_sem = float(df_eval["similaridade_semantica"].mean())

resumo_metricas = pd.DataFrame([
    {"metrica": "ROUGE-L (F1)", "media": round(media_rouge, 4)},
    {"metrica": "Similaridade semantica (cosseno)", "media": round(media_sem, 4)},
])

display(Markdown("### Medias gerais"))
display(resumo_metricas)

In [ ]:
# 3.3 - LLM-as-a-judge (Likert 1 a 5) + distribuicao das notas
import re
import matplotlib.pyplot as plt

def extrair_json(texto):
    match = re.search(r"\{.*\}", texto, flags=re.DOTALL)
    if not match:
        raise ValueError("JSON nao encontrado na resposta do juiz")
    return json.loads(match.group(0))

prompt_juiz_template = dedent("""
Voce atuara como avaliador de respostas de um sistema RAG.
Avalie em escala Likert de 1 a 5 os criterios:
- fidelidade_contexto
- relevancia
- completude

Regras:
- Use apenas o contexto fornecido.
- Seja rigoroso com alucinacoes.
- Retorne SOMENTE JSON valido no formato:
{{
  "fidelidade_contexto": <int>,
  "relevancia": <int>,
  "completude": <int>,
  "justificativa_curta": "..."
}}

Pergunta: {pergunta}

Contexto recuperado:
{contexto}

Resposta gerada:
{resposta}
""")

avaliacoes = []
for _, row in df_eval.iterrows():
    prompt = prompt_juiz_template.format(
        pergunta=row["pergunta"],
        contexto=row["contexto_recuperado"],
        resposta=row["resposta_gerada"],
    )
    bruto = chat_groq(
        prompt,
        system_prompt="Voce e um avaliador tecnico e retorna apenas JSON.",
        temperature=0.0,
        top_p=1.0,
        max_tokens=220,
    )

    try:
        j = extrair_json(bruto)
    except Exception:
        j = {
            "fidelidade_contexto": 1,
            "relevancia": 1,
            "completude": 1,
            "justificativa_curta": "Falha de parsing do avaliador.",
        }

    avaliacoes.append({
        "pergunta": row["pergunta"],
        "fidelidade_contexto": int(j["fidelidade_contexto"]),
        "relevancia": int(j["relevancia"]),
        "completude": int(j["completude"]),
        "justificativa_curta": j.get("justificativa_curta", ""),
    })

df_judge = pd.DataFrame(avaliacoes)
display(Markdown("### Notas do LLM-as-a-judge"))
display(df_judge)

plt.figure(figsize=(10, 4))
for i, col in enumerate(["fidelidade_contexto", "relevancia", "completude"], start=1):
    plt.subplot(1, 3, i)
    df_judge[col].value_counts().sort_index().plot(kind="bar")
    plt.title(col)
    plt.xlabel("nota")
    plt.ylabel("freq")
    plt.ylim(0, max(1, df_judge[col].value_counts().max() + 1))

plt.tight_layout()
plt.show()

display(Markdown("### Medias do LLM-as-a-judge"))
display(df_judge[["fidelidade_contexto", "relevancia", "completude"]].mean().to_frame("media").T)

### Avaliacao Final da Qualidade [PREENCHER MANUALMENTE]

Com base nas metricas e no LLM-as-a-judge, registre sua analise final:

- Casos em que o RAG respondeu com alta fidelidade ao contexto.
- Casos problematicos (ruido de retrieval, falta de completude, resposta incompleta).
- Decisao sobre aceitacao do sistema para uso pratico.
- Melhorias propostas (prompt, chunking, reranking, ajuste de k, ajustes de avaliacao).

## Entrega

| Item | Detalhe |
|------|---------|
| **Formato** | Arquivo `.ipynb` com todas as células executadas |
| **Nome do arquivo** | `trabalho_final_[seu_nome].ipynb` |
| **Prazo** | Conforme comunicado pelo professor |
| **Plataforma** | Submissão via google classroom |

> Notebooks com células não executadas, sem outputs ou que gerem erros ao rodar serão penalizados.
> A presença de API keys expostas diretamente no código implica desconto de **1 ponto**.
